## Part 2: Groq + RAG 
## Task 3: Groq-Based RAG Pipeline

In [13]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import SentenceTransformer
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
load_dotenv()

True

In [24]:

from langchain_core.documents import Document

documents = [
    Document(page_content="""
    Python is a high-level, interpreted programming language known for its simple syntax and readability.
    It is widely used for web development, automation, data analysis, machine learning, and artificial intelligence.
    Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming.
    Popular Python libraries include NumPy, Pandas, Matplotlib, Scikit-learn, TensorFlow, and PyTorch.
    Python developers can build web applications using frameworks such as Django, Flask, and FastAPI.
    """),

    Document(page_content="""
    LangChain is a framework for developing applications powered by large language models.
    It provides components for connecting language models with prompts, external data sources, tools, and APIs.
    LangChain supports prompt templates, output parsers, chains, retrievers, and document loaders.
    Developers can use RunnableSequence to execute components sequentially and RunnableParallel to execute
    multiple operations in parallel.
    LangChain also supports retrieval-augmented generation, conversational AI, and question-answering systems.
    """),

    Document(page_content="""
    RAG stands for Retrieval-Augmented Generation. It is a technique that combines information retrieval
    with the text-generation capabilities of a large language model.
    In a RAG application, documents are divided into smaller chunks and converted into vector embeddings.
    These embeddings are stored in a vector database. When a user asks a question, the system retrieves
    relevant document chunks and passes them to the language model as context.
    The model then generates an answer based on the retrieved information.
    RAG can reduce unsupported answers and help language models answer questions using private or updated data.
    The quality of a RAG system depends on document quality, chunking, embedding models, retrieval accuracy,
    and the instructions given to the language model.
    """),

    Document(page_content="""
    Chroma is a vector database used to store, manage, and search vector embeddings.
    It is commonly used in applications involving semantic search and retrieval-augmented generation.
    Chroma stores embeddings along with associated text and metadata.
    When a query is submitted, Chroma can find documents whose embeddings are similar to the query embedding.
    Developers can use Chroma with LangChain to create retrievers for question-answering applications.
    A Chroma collection can contain multiple documents, and each document can have a unique identifier.
    Chroma supports persistent storage, allowing data to be saved and reused between application sessions.
    """),

    Document(page_content="""
    Groq provides fast inference for large language models through its API and specialized inference infrastructure.
    Developers can access supported language models using an API key.
    Groq can be integrated with LangChain to generate text, answer questions, summarize documents,
    and build conversational applications.
    The ChatGroq integration allows developers to configure a model and parameters such as temperature.
    Temperature influences the variability of generated responses.
    A lower temperature generally produces more predictable output, while a higher temperature can produce
    more varied responses. The models available through Groq depend on its current model offerings.
    """),

    Document(page_content="""
    Large Language Models, commonly called LLMs, are machine learning models trained on large amounts of text.
    They can perform tasks such as answering questions, summarizing content, translating languages,
    generating code, and following natural-language instructions.
    LLMs generate responses based on patterns learned during training and the context provided at inference time.
    Their answers can sometimes contain incorrect information, so important results should be verified.
    Examples of LLM providers and model families include OpenAI models, Google Gemini, and open-weight models
    such as Llama and Qwen.
    """),

    Document(page_content="""
    Embeddings are numerical representations of text, images, or other data.
    Text embeddings represent semantic information as vectors in a multidimensional space.
    Texts with similar meanings may have embeddings that are close to each other.
    Embeddings are useful for semantic search, recommendation systems, clustering, and document retrieval.
    In a RAG application, an embedding model converts document chunks and user queries into vectors.
    A vector database compares these vectors to retrieve potentially relevant content.
    Embedding quality and the choice of model can affect retrieval performance.
    """),

    Document(page_content="""
    A prompt template is a reusable structure for providing instructions and input variables to a language model.
    In LangChain, PromptTemplate is commonly used for text prompts, while ChatPromptTemplate creates
    structured messages for chat models.
    Input variables are supplied when formatting the template.
    System messages define the assistant's behavior, human messages contain user input, and AI messages
    represent assistant responses in a conversation.
    Clear instructions and relevant context help a model generate more useful and consistent answers.
    Prompt templates are frequently used in summarization, question answering, and RAG applications.
    """),

    Document(page_content="""
    A retriever is a component that searches for relevant documents in response to a query.
    In LangChain, retrievers can work with vector databases and other information sources.
    A similarity search compares the query embedding with stored document embeddings.
    The retriever returns the documents considered most relevant to the query.
    These documents can then be combined into context for a language model.
    Retrieval quality can be improved through appropriate chunk sizes, overlap, metadata filtering,
    query rewriting, and reranking techniques.
    """),

    Document(page_content="""
    FastAPI is a Python web framework for building APIs.
    It supports asynchronous programming, request validation, and automatic interactive API documentation.
    FastAPI uses Python type hints and integrates with Pydantic for data validation.
    Developers commonly run FastAPI applications using Uvicorn.
    It can expose endpoints that accept questions, retrieve relevant documents, and return answers generated
    by a language model.
    FastAPI is suitable for integrating machine learning and RAG applications into web services.
    """),

    Document(page_content="""
    Text splitting is the process of dividing long documents into smaller pieces called chunks.
    Chunking is important in RAG because language models have context limits and retrieval systems work
    with smaller, more focused sections of text.
    Chunk size determines how much text each chunk contains, while chunk overlap preserves some text
    between neighboring chunks.
    If chunks are too small, they may lose important context. If chunks are too large, retrieval may include
    unnecessary information.
    Choosing an appropriate chunk size and overlap depends on the document structure and application.
    """),

    Document(page_content="""
    Output parsers help convert language model responses into a format that an application can use.
    LangChain provides output parsers such as StrOutputParser and structured output parsers.
    StrOutputParser extracts text from supported model outputs and returns a string.
    Structured output techniques can produce data that follows a defined schema, such as a Pydantic model.
    Output parsing is useful when an application needs predictable results for displaying answers,
    processing data, or passing information to another component in a chain.
    """)
]

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 100,
    chunk_overlap = 10
)

chunks = text_splitter.split_documents(documents)
print("Total chunks:", len(chunks))
print(chunks[0].page_content)


## create embedding and store chunks in chroma

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

# print("Documents stored in Chroma successfully!")

##  Retrieve Relevant document chunk
retriever = vector_store.as_retriever(
    search_kwargs={"k": 2}
)
# question = 'What is Chroma used for?'

# question = 'Who is Elon Musk'

questions = [
    "What is Chroma used for?",
    "What is RAG?",
    "Who is Elon Musk?"
]

for question in questions:
    retrieved_docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in  retrieved_docs)

# reterived_docs = retriever.invoke(question)

# print(reterived_docs)
## print(reterived_docs)

## Construct the RAG Prompt
# context = "\n\n".join(doc.page_content for doc in  retrieved_docs)

# print(context)

    prompt_template = ChatPromptTemplate.from_template(
        template="""You are a helpful assistant.
    Answer the question using only the provided context.
    If the answer is not present in the context, say:
    "No relevant information found."

    Context: {context}
    Question: {question}
    Answer:"""

    )


    final_prompt = prompt_template.invoke({"context":context, "question":question})
    # print(final_prompt)


## Pass the prompt to the groq
    model = ChatGroq(
        model="openai/gpt-oss-120b",
        temperature=0
    )

    response = model.invoke(final_prompt)
    print(response.content)


Total chunks: 124
Python is a high-level, interpreted programming language known for its simple syntax and


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6717.88it/s]


Chroma is used to store, manage, and search vector embeddings.
RAG stands for Retrieval‑Augmented Generation, a technique that combines information.
No relevant information found.


In [25]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import SentenceTransformer
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
load_dotenv()

True

In [26]:
prompt_template = ChatPromptTemplate.from_messages([
    ("system", """
    You are a helpful assistant.
    Answer the user's question using only the provided context.
    Do not use outside knowledge.
    If the answer is not present in the context,
    say: "No relevant information found."
    """),

    ("human", """
    Retrieved Context:
    {context}

    User Question:
    {question}

    Answer:
    """)
])

question = "What is Chroma used for?"
retrieved_docs = retriever.invoke(question)

context = "\n\n".join(
    doc.page_content for doc in retrieved_docs
)


final_prompt = prompt_template.invoke({
    "context": context,
    "question": question
})

response = model.invoke(final_prompt)

print("Question:", question)
print("Retrieved Context:", context)
print("Answer:", response.content)


questions = [
    "What is Chroma used for?",
    "What is RAG?",
    "Who is Elon Musk?"
]

for question in questions:
    retrieved_docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )

    final_prompt = prompt_template.invoke({
        "context": context,
        "question": question
    })

    response = model.invoke(final_prompt)

    print("Question:", question)
    print("Answer:", response.content)
    print("-" * 50)

Question: What is Chroma used for?
Retrieved Context: Chroma is a vector database used to store, manage, and search vector embeddings.

Chroma is a vector database used to store, manage, and search vector embeddings.
Answer: Chroma is used as a vector database for storing, managing, and searching vector embeddings.
Question: What is Chroma used for?
Answer: Chroma is used as a vector database for storing, managing, and searching vector embeddings.
--------------------------------------------------
Question: What is RAG?
Answer: RAG stands for Retrieval‑Augmented Generation. It is a technique that combines information.
--------------------------------------------------
Question: Who is Elon Musk?
Answer: No relevant information found.
--------------------------------------------------
